# gem

> Simple utilities for working with Google's Gemini API

This notebook provides a minimal interface to Google's Gemini API. The goal is to make it dead simple to:

1. Generate text with just a prompt
2. Analyze files (PDFs, images, **MP4 videos**) 
3. Process videos (YouTube URLs or **local MP4 files**)

All through a single `gem()` function that just works.

In [1]:
#| default_exp gem

In [2]:
#| hide
from nbdev.showdoc import *

## Setup

First, make sure you have your Gemini API key set:

In [3]:
#| export
import os, time
from pathlib import Path
from fastcore.all import *
from google import genai
from google.genai import types
from functools import partial
from fastprogress import progress_bar

In [4]:
# export GEMINI_API_KEY='your-api-key'
assert os.environ.get("GEMINI_API_KEY"), "Please set GEMINI_API_KEY environment variable"

## Building blocks

Let's start with the simple helper functions that make everything work.

### Client creation

We need a Gemini client to talk to the API:

In [5]:
#|export
def _client():
    "Get Gemini client"
    return genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

In [6]:
#|hide
c = _client()
assert c is not None
assert hasattr(c, 'models')

## Video upload

In [20]:
#|export
def upload_file(pth):
    if not Path(pth).exists(): raise ValueError(f"File {pth} does not exist.")
    c = _client()
    f = c.files.upload(file=pth)
    time.sleep(2)
    for i in progress_bar(range(15)):
        try:
            f = c.files.get(name=f.name)
            if f.state == 'ACTIVE': return f
            elif f.state == 'FAILED': raise Exception(f'File processing for {pth} failed.')
            time.sleep(10)
        except: pass # because the gemini file thing is jank
    raise Exception(f'Timeout processing {pth}')

In [21]:
myfile = upload_file("_videos/test_video.mp4")
assert myfile.state == 'ACTIVE'

In [22]:
myfile = upload_file("_videos/lesson1_small.mp4")
assert myfile.state == 'ACTIVE'

### Converting attachments to Parts

Gemini expects different types of content (files, URLs) to be wrapped in "Parts". This helper handles that conversion:

In [43]:
#| export
def _is_url(s):
    "Check if string is a URL"
    if not isinstance(s, str): return False
    return (s.startswith('http://') or 
            s.startswith('https://') or 
            s.startswith('www.') or 
            'youtube.com' in s or 
            'youtu.be' in s)

def _make_part(o):
    "Convert object to Gemini Part"
    if isinstance(o, types.File):
        return types.Part.from_uri(file_uri=o.uri, mime_type=o.mime_type)
    if isinstance(o, (str, Path)):
        p = Path(o)
        if p.exists():
            if p.suffix.lower() == '.mp4':
                f = upload_file(o)
                return types.Part.from_uri(file_uri=f.uri, mime_type=f.mime_type)
            mime_map = {'.pdf': 'application/pdf', 
                        '.png': 'image/png', 
                        '.jpg': 'image/jpeg', 
                        '.jpeg': 'image/jpeg', 
                        '.gif': 'image/gif'}
            mime = mime_map.get(p.suffix.lower(), 'application/octet-stream')
            return types.Part.from_bytes(mime_type=mime, data=p.read_bytes())
        elif _is_url(o): return types.Part.from_uri(file_uri=o, mime_type='video/*')
        else: raise ValueError(f"Could not parse file or url: {o}")
    return None

In [44]:
_part = _make_part('_videos/test_video.mp4')
_part

Part(
  file_data=FileData(
    file_uri='https://generativelanguage.googleapis.com/v1beta/files/6qasiq3n6fpv',
    mime_type='video/mp4'
  )
)

## The main interface

Now we can build our main `gem()` function that handles all use cases:

In [48]:
#| export
def gem(prompt, # Text prompt
        o=None, # Optional file/URL attachment or list of attachments
        model='gemini-2.5-flash',
        thinking=-1,
        search=False):
    "Generate content with Gemini"
    parts = [types.Part.from_text(text=prompt)]
    # Handle single attachment or list of attachments
    attachments = o if isinstance(o, list) else [o] if o else []
    for attachment in attachments:
        if part := _make_part(attachment): parts.insert(0, part)
    
    contents = types.Content(role='user', parts=parts) if attachments else prompt    
    config_dict = {
        'thinking_config': types.ThinkingConfig(thinking_budget=thinking),
        'response_mime_type': 'text/plain'
    }
    # Adjust media_resolution for videos for more tokens
    if any(p.file_data and p.file_data.mime_type.startswith('video') for p in parts):
        config_dict['media_resolution'] = 'MEDIA_RESOLUTION_LOW'
    config_dict['tools'] = []
    if search: config_dict['tools'].append(types.Tool(google_search=types.GoogleSearch()))
    cfg = types.GenerateContentConfig(**config_dict)
    resp = _client().models.generate_content(model=model, contents=contents, config=cfg)
    return resp.text

## Examples

One function handles everything:
- Just text? Pass a prompt.
- Have a file? Pass it as the second argument.
- Got a YouTube URL? Same thing.

Let's test it out:

## Text generation

The simplest case - just generate some text:

In [27]:
gem("Write a haiku about Python programming")

'Simple, clean to type,\nIndented, logic unfolds,\nPowerful, so swift.'

## Video analysis

Perfect for creating YouTube chapters or summaries:

In [29]:
prompt = "5 word summary of this video."
gem(prompt, "https://youtu.be/1x3k0V2IITo")

'Late interaction overcomes single vector limits.'

### Local MP4 Video Analysis

You can also analyze local MP4 video files:

In [28]:
# Example with local MP4 file (if you have one)
gem("Summarize this video in 3 sentences.", "_videos/lesson1_small.mp4")

'This course aims to teach participants how to systematically evaluate LLM-powered products by mastering principles like error analysis and automated evaluators. It addresses the inherent difficulty of LLM development, characterized by "Three Gulfs"—comprehension of data, specification of precise instructions, and generalization to new inputs—requiring diverse skill sets. The course guides users through an iterative "Analyze-Measure-Improve" lifecycle, emphasizing the importance of understanding user unhappiness and effective prompting to ensure reliable, safe, and useful AI.'

### File analysis

Great for extracting information from PDFs or images:

In [30]:
gem("3 sentence summary of this presentation.", "NewFrontiersInIR.pdf")

'This presentation explores "New Frontiers in IR," highlighting how traditional keyword and semantic search methods fall short when users provide complex instructions or queries requiring multi-step reasoning. To address this, two models are introduced: Promptriever, a fast bi-encoder trained to follow instructions and be "promptable" like an LLM, and Rank1, a strong but slower cross-encoder that incorporates test-time reasoning. Evaluations show significant improvements for both, demonstrating that IR systems can achieve higher recall, greater robustness to prompts, and solve more intricate information needs by adopting these instruction-following and reasoning capabilities.'

In [31]:
gem("What's in this image?", "anton.png")

'This image appears to be a YouTube video thumbnail, designed to be visually engaging and informative about a technical topic, likely related to AI or data processing.\n\nHere\'s a detailed description of what\'s in the image:\n\n*   **Background:** A solid, dark blue or almost black background provides high contrast for the foreground elements.\n*   **Text:**\n    *   At the top left, in large white letters, is the question: "Single Vector?".\n    *   Below it, in even larger, bright yellow capital letters, is the phrase: "YOU\'RE MISSING OUT".\n*   **Person:** On the left side of the image, positioned slightly below the "Single Vector?" text, is a smiling young man with light brown hair, looking directly at the viewer. He is wearing a white t-shirt. His head and upper shoulders are visible.\n*   **Emoji:** Just above and slightly to the left of the yellow "YOU\'RE" text, a yellow "sad" or "disappointed" emoji (with downturned mouth and eyebrows) is placed, partially overlapping the m

### Change Model

You can also control the model and thinking time:

In [32]:
gem("What is Hamel Husain's current job?", model="gemini-2.5-pro")

'Hamel Husain is the **co-founder and CEO** of **Outerbounds**.\n\nOuterbounds is a company focused on MLOps (Machine Learning Operations) and provides an enterprise platform built around the popular open-source framework **Metaflow**, which was originally developed at Netflix.\n\nBefore co-founding Outerbounds in 2021, he was a Principal Machine Learning Scientist at GitHub.'

### Grounded Search

As you can see, grounded search is required to get things right sometimes!

In [33]:
gem("What is Hamel Husain's current job?.", search=True)

'Hamel Husain is currently an independent consultant, assisting companies with building AI products, particularly focusing on large language models (LLMs). He is also the founder of Parlance Labs.\n\nPrior to his current role, he was a Staff Machine Learning Engineer at GitHub. He has over 25 years of experience in machine learning, having also worked with companies such as Airbnb and DataRobot. Additionally, he teaches a course on "AI Evals For Engineers & PMs".'

### Multiple Attachments

You can analyze multiple files/URLs at once by passing a list:

In [34]:
prompt = "Is this PDF and YouTube video related or are they different talks? Answer with very short yes/no answer."
gem(prompt, ["https://youtu.be/Trps2swgeOg?si=yK7CO0Zk4E1rfp6s", "NewFrontiersInIR.pdf"])

'No'

In [35]:
gem(prompt, ["https://youtu.be/YB3b-wPbSH8?si=WI0LqflY5SYIsRz9", "NewFrontiersInIR.pdf"])

'Yes.'

In [49]:
gem("What do these slides and this video have in common in terms of content/subject matter if at all? Provide a 1 sentence summary of each.", ["NewFrontiersInIR.pdf", "_videos/test_video.mp4"])

'The video and the slides do not share common content or subject matter beyond the general concept of "testing."\n\n**Video Summary:** This video is a brief test recording featuring a man speaking his name and reciting numbers to check audio quality.\n\n**Slides Summary:** This presentation introduces "Promptriever" and "Rank1," novel information retrieval models designed to follow natural language instructions and perform reasoning, proposing a shift from traditional search methods by leveraging instruction-trained retrievers and test-time computation.'

## Export -

In [51]:
#| hide
import nbdev; nbdev.nbdev_export()